<a href="https://colab.research.google.com/github/rantawadeesritakorn-tech/Project-Hotel/blob/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88-2/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ส่วนที่ 4 — ฟังก์ชันช่วยงานและ class Booking

ฟังก์ชันทั้งหมด 15 ฟังก์ชัน มี parameter และ return
และมี default argument 10 ฟังก์ชัน

In [1]:
import random
import math

def pick_nationality():
    "สุ่มสัญชาติตามสัดส่วนตลาดจริงของนักท่องเที่ยวที่เข้าไทย"
    return random.choices(NATIONALITIES, weights=NATIONALITY_WEIGHTS, k=1)[0]

def generate_guest_name(nationality="Thai"):
    "สุ่มชื่อขนามสกุลให้ตรงกับสัญชาติ"
    p = MARKET_PROFILE[nationality]
    return f"{random.choice(p['first_names'])} {random.choice(p['last_names'])}"

def generate_phone(nationality="Thai", prefix="08"):
    "สุ่มเบอร์โทร"
    country_code = {"Chinese": "+86", "Malaysian": "+60", "Indian": "+91",
                    "Russian": "+7", "Korean": "+82", "Japanese": "+81",
                    "British": "+44", "German": "+49", "American": "+1",
                    "Singaporean": "+65", "Australian": "+61"}
    digits = "".join(str(random.randint(0, 9)) for _ in range(8))
    if nationality == "Thai":
        return prefix + digits
    return f"{country_code[nationality]}{digits}"

def pick_travel_party():
    "สุ่มลักษณะกลุ่มผู้เข้าพัก"
    kinds = list(TRAVEL_PARTY.keys())
    weights = [TRAVEL_PARTY[k]["weight"] for k in kinds]
    kind = random.choices(kinds, weights=weights, k=1)[0]
    info = TRAVEL_PARTY[kind]
    children = random.randint(1, 2) if info["children"] else 0
    return kind, info["adults"], children

def pick_room_type(party_kind="couple"):
    "สุ่มประเภทห้องตามลักษณะกลุ่มผู้เข้าพัก"
    w = TRAVEL_PARTY[party_kind]["room_weights"]
    return random.choices(list(w.keys()), weights=list(w.values()), k=1)[0]

def pick_channel(nationality="Thai", lead_time_days=0):
    "สุ่มช่องทางการจองตามพฤติกรรมของแต่ละสัญชาติ"
    channels = ["OTA", "Website", "Walk-in", "Phone"]
    weights = list(MARKET_PROFILE[nationality]["channel_weights"])
    if lead_time_days > 1:
        weights[2] = 0
    return random.choices(channels, weights=weights, k=1)[0]

def random_nights(nationality="Thai", max_nights=14):
    """สุ่มจำนวนคืนแบบ geometric รอบค่าเฉลี่ยของสัญชาตินั้น -- มี DEFAULT ARGUMENT

    การกระจายแบบนี้ทำให้ได้ผลตรงกับสถิติจริง: การจองส่วนใหญ่ (~2 ใน 3)
    เป็นการพัก 1-2 คืน แต่ยังมีหางยาวของคนที่พัก 7 คืนขึ้นไป
    """
    mean = MARKET_PROFILE[nationality]["avg_nights"]
    p = 1 / mean
    n = 1
    while random.random() > p and n < max_nights:
        n += 1
    return n

def random_lead_time(nationality="Thai", max_days=180):
    "สุ่มจำนวนวันที่จองล่วงหน้า"
    mean = MARKET_PROFILE[nationality]["lead_time"]
    days = int(random.expovariate(1 / mean))
    return min(days, max_days)

def wants_breakfast(nationality="Thai"):
    "สุ่มว่าซื้ออาหารเช้าไหม ตามพฤติกรรมของแต่ละสัญชาติ"
    return random.random() < MARKET_PROFILE[nationality]["breakfast"]

def demand_factor(check_in):
    "ตัวคูณดีมานด์ของวันนั้น"
    return MONTH_DEMAND[check_in.month] * WEEKDAY_DEMAND[check_in.weekday()]

def seasonal_multiplier(check_in, high=1.25, low=0.90):
    "ตัวคูณราคาตามฤดูกาล -- มี DEFAULT ARGUMENT (high=1.25, low=0.90)"
    if check_in.month in HIGH_SEASON_MONTHS:
        return high
    if check_in.month in LOW_SEASON_MONTHS:
        return low
    return 1.00

def dynamic_rate_multiplier(occupancy, threshold_high=0.85, threshold_mid=0.70):
    "Revenue management"
    if occupancy >= threshold_high:
        return 1.20
    if occupancy >= threshold_mid:
        return 1.10
    if occupancy < 0.40:
        return 0.92
    return 1.00

def classify_season(check_in):
    "แปลงเดือนเป็นชื่อฤดูกาลแบบข้อความ"
    if check_in.month in HIGH_SEASON_MONTHS:
        return "High"
    if check_in.month in LOW_SEASON_MONTHS:
        return "Low"
    return "Normal"

def poisson(lam):
    "สุ่มจำนวนเหตุการณ์ต่อวันแบบ Poisson"
    limit, k, p = math.exp(-lam), 0, 1.0
    while True:
        p *= random.random()
        if p <= limit:
            return k
        k += 1

def daterange(start=SIM_START, end=SIM_END):
    "วนทีละวันจาก start ถึง end"
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

NameError: name 'SIM_START' is not defined

In [ ]:
class Booking:
    "ใบจอง 1 ใบ - คำนวณราคาทันทีตอนสร้าง แล้วเก็บผลไว้เป็น attribute"

    def __init__(self, booking_id, guest, room, check_in, nights, channel,
                 booking_date, breakfast=False, upgraded=False,
                 adults=2, children=0, rate_multiplier=1.0):
        self.booking_id = booking_id
        self.guest = guest
        self.room = room
        self.check_in = check_in
        self.nights = nights
        self.check_out = check_in + timedelta(days=nights)
        self.channel = channel
        self.booking_date = booking_date
        self.lead_time_days = (check_in - booking_date).days
        self.breakfast = breakfast
        self.upgraded = upgraded
        self.adults = adults
        self.children = children
        self.rate_multiplier = rate_multiplier
        self.status = "CONFIRMED"
        self.cancel_date = None
        self.tier_at_booking = guest.member_tier
        self.discount_rate_used = guest.discount_rate()
        self.price_detail = self.calculate_price()

    def calculate_price(self):
        "คำนวณราคาทั้งใบ แล้ว return เป็น dict (แยกให้เห็นทุกองค์ประกอบ)"
        nightly = self.room.price_per_night(self.check_in) * self.rate_multiplier
        room_charge = round(nightly, 2) * self.nights
        extra_charge = BREAKFAST_PRICE * self.nights * self.adults if self.breakfast else 0
        subtotal = room_charge + extra_charge
        discount = subtotal * self.discount_rate_used
        net = subtotal - discount
        service_charge = net * SERVICE_CHARGE_RATE
        vat = (net + service_charge) * VAT_RATE
        total = net + service_charge + vat
        commission = total * CHANNEL_COMMISSION[self.channel]
        return {
            "rate_per_night": round(nightly, 2),
            "room_charge": round(room_charge, 2),
            "extra_charge": round(extra_charge, 2),
            "discount": round(discount, 2),
            "service_charge": round(service_charge, 2),
            "vat": round(vat, 2),
            "total_price": round(total, 2),
            "commission": round(commission, 2),
            "net_revenue": round(total - commission, 2),
        }

    def cancel(self, cancel_date=None):
        "ลูกค้ายกเลิก"
        self.status = "CANCELLED"
        self.cancel_date = cancel_date or self.booking_date
        self.room.release(self.check_in, self.nights)

    def mark_no_show(self):
        "ลูกค้าไม่มาเช็คอินโดยไม่แจ้งล่วงหน้า - โรงแรมยังเก็บเงินคืนแรกได้"
        self.status = "NO_SHOW"

    def check_out_guest(self):
        "เช็คเอาท์เรียบร้อย"
        self.status = "CHECKED_OUT"

    def is_revenue(self):
        "ใบจองนี้นับเป็นรายได้จริงไหม (ยกเลิกแล้วไม่นับ)"
        return self.status in ("CONFIRMED", "CHECKED_OUT", "NO_SHOW")

    def to_dict(self):
        "แปลง object"
        row = {
            "booking_id": self.booking_id,
            "guest_id": self.guest.guest_id,
            "room_id": self.room.room_id,
            "room_number": self.room.room_number,
            "room_type": self.room.room_type,
            "booking_date": self.booking_date.isoformat(),
            "check_in": self.check_in.isoformat(),
            "check_out": self.check_out.isoformat(),
            "nights": self.nights,
            "lead_time_days": self.lead_time_days,
            "adults": self.adults,
            "children": self.children,
            "booking_month": self.check_in.strftime("%Y-%m"),
            "weekday": self.check_in.strftime("%A"),
            "season": classify_season(self.check_in),
            "channel": self.channel,
            "nationality": self.guest.nationality,
            "breakfast": int(self.breakfast),
            "upgraded": int(self.upgraded),
            "member_tier_at_booking": self.tier_at_booking,
            "status": self.status,
            "cancel_date": self.cancel_date.isoformat() if self.cancel_date else "",
        }
        row.update(self.price_detail)
        return row

    def __repr__(self):
        return (f"Booking({self.booking_id}, {self.guest.name}, "
                f"{self.room.room_number}, {self.nights} คืน, {self.status})")